# Muse EEG Heads — Sleep-EDF Windows

Focused pilot: Sleep-EDF Expanded → Muse-proxy tensors → Head A vigilance windows.

**Channels proxy:** AF7=AF8=Fpz-Cz, TP9=TP10=Pz-Oz @ 256 Hz (bandpass 1–45 Hz).

**Labels:** W→`drowsy`, N1→`hypnagogic`; exclude N2+/REM/?/Movement.

**License:** open mix only — **NO LUNA**, **NO L-FAME (BY-NC)**, **NO SEED-VIG**. Sleep-EDF is ODC-By (PhysioNet). See `docs/LICENSE_NOTES.md`.

Companion outline kernel: `muse-eeg-heads-cbramod-outline` (cache/download + stubs).


## 1. Setup

Helpers from private dataset **`windwerfer/muse-eeg-heads-src`**. Prefer `uv` when installing.


In [ ]:
# Setup — load helpers from Kaggle Dataset windwerfer/muse-eeg-heads-src
import os, sys, random, shutil, subprocess
from pathlib import Path

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
INPUT_SRC = Path("/kaggle/input/muse-eeg-heads-src")
SRC = WORKING / "src"
SRC.mkdir(parents=True, exist_ok=True)

if INPUT_SRC.exists():
    for f in INPUT_SRC.glob("*.py"):
        shutil.copy(f, SRC / f.name)
    print("loaded modules from", INPUT_SRC, "→", SRC)
else:
    local = Path(".").resolve()
    for cand in (local / "src", local.parent / "src", Path("/workspace/muse-eeg-heads/src")):
        if cand.exists():
            for f in cand.glob("*.py"):
                shutil.copy(f, SRC / f.name)
            print("loaded modules from", cand)
            break
    else:
        raise FileNotFoundError(
            "Missing muse-eeg-heads-src dataset. "
            "Add data source windwerfer/muse-eeg-heads-src."
        )

sys.path.insert(0, str(WORKING))
ROOT = WORKING

need = []
for mod, pipname in [("numpy", "numpy")]:
    try:
        __import__(mod)
    except ImportError:
        need.append(pipname)
if need:
    if subprocess.call(["bash", "-lc", "command -v uv >/dev/null"]) == 0:
        subprocess.check_call(["uv", "pip", "install", "--system", "-q", *need])
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("ROOT", ROOT)
from src.channel_map import MUSE_CHANNELS
from src.metrics import HEAD_A_LABELS
print("MUSE", MUSE_CHANNELS)
print("Head A", HEAD_A_LABELS)


## 2. Seed cache

Copy Sleep-EDF pilot from **`windwerfer/muse-eeg-heads-cache`**. Optional CBraMod path print only.


In [ ]:
# Seed Sleep-EDF pilot from muse-eeg-heads-cache
import shutil
from pathlib import Path

DATA = Path("/kaggle/working/data") if Path("/kaggle/working").exists() else Path("/workspace/muse-eeg-heads/kaggle_datasets/muse-eeg-heads-cache/data")
MODELS = Path("/kaggle/working/models") if Path("/kaggle/working").exists() else Path("/workspace/muse-eeg-heads/kaggle_datasets/muse-eeg-heads-cache/models")
CACHE = Path("/kaggle/input/muse-eeg-heads-cache")
LOCAL_CACHE = Path("/workspace/muse-eeg-heads/kaggle_datasets/muse-eeg-heads-cache")

if not CACHE.exists() and LOCAL_CACHE.exists():
    CACHE = LOCAL_CACHE
    print("using local cache mirror:", CACHE)

DATA.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

PILOT = DATA / "sleep-edfx-pilot"
if CACHE.exists():
    src = CACHE / "data" / "sleep-edfx-pilot"
    if src.exists():
        if PILOT.exists() and not Path("/kaggle/working").exists():
            # local box: point at cache in place
            PILOT = src
            print("local pilot path", PILOT)
        else:
            if PILOT.exists():
                shutil.rmtree(PILOT)
            shutil.copytree(src, PILOT)
            print(f"cached Sleep-EDF pilot: {len(list(PILOT.rglob('*.edf')))} edf files → {PILOT}")
    cbr = CACHE / "models" / "CBraMod" / "pretrained_weights.pth"
    if cbr.exists():
        print(f"CBraMod weights present: {cbr} ({cbr.stat().st_size/1e6:.1f} MB)")
    else:
        print("CBraMod weights not in cache (optional for this notebook)")
else:
    raise FileNotFoundError("Attach windwerfer/muse-eeg-heads-cache (or local mirror)")

assert list(PILOT.rglob("*.edf")), f"No EDF under {PILOT}"
print("cache seed OK")


## 3. Load first night around N1

`around_stage="stage 1"`, `pre_sec=20*60`, `post_sec=40*60` → Muse-proxy `(4, T)` @ 256 Hz.


In [ ]:
# Load first Sleep-EDF night around first N1
from pathlib import Path
from collections import Counter
from src.sleep_edf import find_pilot_pairs, load_sleep_edf_recording, PROXY_NOTE
from src.channel_map import MUSE_CHANNELS

TARGET_SR = 256
pairs = find_pilot_pairs(PILOT)
assert pairs, f"No PSG/Hypnogram pairs under {PILOT}"
print(f"found {len(pairs)} night(s)")
for psg, hyp in pairs:
    print(" ", psg.name, "+", hyp.name)

psg_path, hyp_path = pairs[0]
rec = load_sleep_edf_recording(
    psg_path,
    hyp_path,
    target_sr=TARGET_SR,
    around_stage="stage 1",
    pre_sec=20 * 60,
    post_sec=40 * 60,
)
x = rec["data"]  # (4, T)
assert rec["ch_names"] == MUSE_CHANNELS
print(PROXY_NOTE)
print("slice_start_sec", rec["slice_start_sec"])
print("EEG shape", x.shape, "sfreq", rec["sfreq"])
print("stage counts", Counter(rec["stages"].tolist()))
print("preprocess OK")


## 4. Windowing + Head A labels

2.0 s windows, 0.5 s hop. Majority stage ≥70% → label; else drop.

| Sleep-EDF stage | Head A |
|-----------------|--------|
| W | `drowsy` |
| 1 (N1) | `hypnagogic` |
| 2/3/4/R/?/Movement | exclude |


In [ ]:
# Windowing + Head A vigilance labels
from collections import Counter
from src.sleep_edf import windows_with_head_a_labels
from src.metrics import HEAD_A_LABELS

WINDOW_SEC = 2.0
HOP_SEC = 0.5

wins_a, labs_a, keep_idx = windows_with_head_a_labels(
    x,
    rec["stages"],
    sfreq=TARGET_SR,
    window_sec=WINDOW_SEC,
    hop_sec=HOP_SEC,
    majority_frac=0.7,
)
counts = Counter(labs_a)
print("Head A windows", wins_a.shape, "label counts", counts)
assert wins_a.ndim == 3 and wins_a.shape[1] == 4
assert set(labs_a) <= set(HEAD_A_LABELS)
assert "hypnagogic" in labs_a or "drowsy" in labs_a, "expected at least one vigilance label"
if "drowsy" in counts and "hypnagogic" in counts:
    print("both drowsy and hypnagogic present — good")
else:
    print("warning: only one of drowsy/hypnagogic present:", counts)
print("windowing OK")


## 5. Save artifacts

`/kaggle/working/artifacts/sleep_edf_pilot_windows.npz` + small JSON manifest.


In [ ]:
# Save windows + labels + manifest
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np

OUT = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("/workspace/muse-eeg-heads/artifacts")
OUT.mkdir(parents=True, exist_ok=True)
npz_path = OUT / "sleep_edf_pilot_windows.npz"
np.savez_compressed(
    npz_path,
    windows=wins_a.astype(np.float32),
    labels=np.asarray(labs_a),
    keep_idx=np.asarray(keep_idx, dtype=np.int32),
)
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "psg": Path(rec["psg_path"]).name,
    "hypnogram": Path(rec["hypno_path"]).name,
    "sfreq": float(rec["sfreq"]),
    "slice_start_sec": float(rec["slice_start_sec"]),
    "window_sec": WINDOW_SEC,
    "hop_sec": HOP_SEC,
    "majority_frac": 0.7,
    "n_windows": int(wins_a.shape[0]),
    "window_shape": list(wins_a.shape),
    "label_counts": dict(Counter(labs_a)),
    "channels": list(rec["ch_names"]),
    "proxy_note": PROXY_NOTE,
    "around_stage": "stage 1",
    "pre_sec": 20 * 60,
    "post_sec": 40 * 60,
    "npz": str(npz_path.name),
}
man_path = OUT / "sleep_edf_pilot_windows_manifest.json"
man_path.write_text(json.dumps(manifest, indent=2))
print("saved", npz_path, f"({npz_path.stat().st_size/1e6:.2f} MB)")
print("manifest", man_path)
print(json.dumps(manifest, indent=2))


## 6. Optional stub encoder fit

Tiny placeholder — real CBraMod forward lives in the outline / later training kernels.


In [ ]:
# Optional tiny stub encoder fit (short)
import numpy as np
from src.metrics import HEAD_A_LABELS

HEAD_A = list(HEAD_A_LABELS)
label_to_id = {n: i for i, n in enumerate(HEAD_A)}
n_take = min(256, len(labs_a))
Xb = wins_a[:n_take]
ya = np.asarray([label_to_id[l] for l in labs_a[:n_take]], dtype=int)
# stub "encoder" — zeros; print fit summary only
Z = np.zeros((n_take, 256), dtype=np.float32)
print(f"[stub] encoder dim={Z.shape[1]} on {n_take} windows")
print("  y counts:", {HEAD_A[i]: int((ya == i).sum()) for i in range(len(HEAD_A)) if (ya == i).any()})
print("sleep-edf windows notebook done")
